# Canopée agrégée au carroyage 200 m (SITG AGGLO_CARREAU_200) — Grand Genève

Ce notebook :

1. lit une couche **vectorielle de canopée** (GeoPackage, polygones) ;
2. la reprojette en **CH1903+/LV95 (EPSG:2056)** et corrige les géométries invalides ;
3. calcule, pour chaque carreau de 200 m, **l'aire couverte par la canopée**
   (les polygones qui se chevauchent ne sont comptés qu'une fois) ;
4. en tire deux scores :
   - `canopee_part` : part de l'aire du carreau sous canopée, **entre 0 et 1**
     (0 = aucun arbre, 1 = carreau entièrement couvert) ;
   - `canopee_seuil` : version binaire, **1 si `canopee_part` ≥ `SEUIL`** (50 % par défaut), sinon 0 ;
5. exporte les **deux couches** en GeoPackage (un fichier, deux couches) ainsi qu'en Parquet et CSV.

> La part d'aire est déjà normalisée entre 0 et 1 par construction : pas besoin de
> min-max, qui rendrait le score dépendant du carreau le plus boisé du jeu de données.

> ⚠️ **Emprise des données** : si la couche de canopée ne couvre qu'une partie du
> Grand Genève (par ex. le canton de Genève seulement), les carreaux hors couverture
> recevraient un faux 0. Renseigne alors `EMPRISE_CANOPEE` : le score est calculé
> sur la seule partie couverte du carreau, et les carreaux trop peu couverts sont
> mis à vide.

## Dépendances

In [33]:
import importlib.util
import subprocess
import sys

REQUIRED = ["geopandas", "shapely", "pyproj", "pyarrow", "numpy", "pandas", "pyogrio", "rasterio"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely

print("geopandas", gpd.__version__, "| shapely", shapely.__version__, "| numpy", np.__version__)

geopandas 1.0.1 | shapely 2.0.6 | numpy 2.0.0


## Paramètres

- `CANOPEE_GPKG` / `CANOPEE_LAYER` : fichier et couche de canopée (`None` = première
  couche de polygones trouvée ; la liste des couches est affichée).
- `CANOPEE_FILTRE` : requête pandas optionnelle pour ne garder qu'une partie des
  objets (ex. `"hauteur >= 3"`), `None` sinon.
- `EMPRISE_CANOPEE` : polygone(s) de couverture des données de canopée, `None` si
  la couche couvre tout le Grand Genève.
- `COUVERTURE_MIN` : part minimale du carreau couverte par les données pour lui
  attribuer un score (utile seulement avec `EMPRISE_CANOPEE`).
- `SEUIL` : seuil de la version binaire (0,5 = 50 % de l'aire du carreau).
- `PERIMETER_VARIANT` : seuls les carreaux qui touchent ce périmètre sont exportés
  (`None` = tout le carroyage).

In [ ]:
CANOPEE_GPKG = "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/output/walk/GG/step-2/gpkg_attributs/canopee.gpkg"          # à adapter
CANOPEE_LAYER = None                     # None = première couche de polygones
CANOPEE_FILTRE = None                    # ex. "hauteur >= 3"

CARREAU_PATH = "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp"  # à adapter
CARREAU_ID = "GRID_ID"

PERIMETER_VARIANTS = {
    "avec_lac": {"kind": "shp", "path": "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp"},
    "sans_lac": {"kind": "parquet", "path": "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/input/network_agreg/AGGLO_PERIMETRE_SANS_LAC-parquet/AGGLO_PERIMETRE_SANS_LAC.parquet"},
}
PERIMETER_VARIANT = "avec_lac"           # None = tout le carroyage

EMPRISE_CANOPEE = None                   # ex. {"kind": "shp", "path": ".../canton_ge.shp"}
COUVERTURE_MIN = 0.5

SEUIL = 0.5                              # canopee_seuil = 1 si canopee_part >= SEUIL

TARGET_CRS = "EPSG:2056"
OUT_DIR = Path("sorties_canopee")
OUT_STEM = "canopee_carreau200"

## Lecture du carroyage et du périmètre

In [35]:
def read_any(spec):
    gdf = gpd.read_file(spec["path"]) if spec["kind"] == "shp" else gpd.read_parquet(spec["path"])
    return gdf.to_crs(TARGET_CRS)


carreaux = gpd.read_file(CARREAU_PATH).to_crs(TARGET_CRS)
print(f"Carroyage : {len(carreaux):,} carreaux")

if PERIMETER_VARIANT is not None:
    perimeter = read_any(PERIMETER_VARIANTS[PERIMETER_VARIANT]).union_all()
    carreaux = carreaux[carreaux.intersects(perimeter)]
    print(f"Périmètre « {PERIMETER_VARIANT} » : {len(carreaux):,} carreaux le touchent")
carreaux = carreaux.reset_index(drop=True)

aire_carreau = carreaux.geometry.area.to_numpy()

Carroyage : 166,762 carreaux
Périmètre « avec_lac » : 56,460 carreaux le touchent


## Lecture de la canopée

On ne lit que l'emprise du carroyage retenu (`bbox`), on reprojette, on corrige les
géométries invalides (fréquentes dans les couches issues de vectorisation) et on
éclate les multi-polygones pour accélérer l'index spatial.

In [36]:
couches = pyogrio.list_layers(CANOPEE_GPKG)
print("Couches du GeoPackage :")
for nom, geom in couches:
    print(f"  {nom:40s} {geom}")

if CANOPEE_LAYER is None:
    poly = [nom for nom, geom in couches if geom and "Polygon" in geom]
    if not poly:
        raise ValueError("Aucune couche de polygones : précise CANOPEE_LAYER (une couche raster n'est pas gérée ici).")
    CANOPEE_LAYER = poly[0]
print(f"\nCouche utilisée : {CANOPEE_LAYER}")

info = pyogrio.read_info(CANOPEE_GPKG, layer=CANOPEE_LAYER)
src_crs = info["crs"]
if src_crs is None:
    raise ValueError("La couche de canopée n'a pas de SCR déclaré.")
bbox = tuple(gpd.GeoSeries([shapely.box(*carreaux.total_bounds)], crs=TARGET_CRS).to_crs(src_crs).total_bounds)

canopee = gpd.read_file(CANOPEE_GPKG, layer=CANOPEE_LAYER, bbox=bbox, engine="pyogrio")
print(f"{len(canopee):,} objets lus dans l'emprise (SCR source : {src_crs})")
if CANOPEE_FILTRE:
    canopee = canopee.query(CANOPEE_FILTRE)
    print(f"{len(canopee):,} objets après filtre « {CANOPEE_FILTRE} »")

canopee = canopee.to_crs(TARGET_CRS)
geoms = canopee.geometry.to_numpy()
geoms = geoms[~shapely.is_empty(geoms) & ~shapely.is_missing(geoms)]
invalides = ~shapely.is_valid(geoms)
if invalides.any():
    print(f"{invalides.sum():,} géométries invalides corrigées (make_valid)")
    geoms[invalides] = shapely.make_valid(geoms[invalides])
geoms = shapely.get_parts(geoms)                                   # éclate multi-polygones et collections
geoms = geoms[shapely.get_type_id(geoms) == 3]                     # polygones seulement
print(f"{len(geoms):,} polygones de canopée, {shapely.area(geoms).sum() / 1e6:,.2f} km² (chevauchements inclus)")

Couches du GeoPackage :
  canopee                                  Polygon

Couche utilisée : canopee
129,164 objets lus dans l'emprise (SCR source : EPSG:4326)
129,164 polygones de canopée, 1,017.73 km² (chevauchements inclus)


## Aire de canopée par carreau

1. un index spatial (`STRtree`) donne les paires (polygone, carreau) qui se touchent ;
2. on calcule l'intersection de chaque paire ;
3. par carreau, on **fusionne** les morceaux avant de mesurer l'aire, pour ne pas
   compter deux fois les zones où des polygones se chevauchent. La fusion n'est
   faite que pour les carreaux qui ont plusieurs morceaux (les autres sont directs).

Le calcul se fait par paquets de polygones pour limiter la mémoire.

In [37]:
cell_geoms = carreaux.geometry.to_numpy()
tree = shapely.STRtree(cell_geoms)

morceaux_geom, morceaux_cell = [], []
PAQUET = 200_000
for start in range(0, len(geoms), PAQUET):
    g = geoms[start:start + PAQUET]
    i_can, i_cell = tree.query(g, predicate="intersects")
    inter = shapely.intersection(g[i_can], cell_geoms[i_cell])
    garde = shapely.area(inter) > 0
    morceaux_geom.append(inter[garde])
    morceaux_cell.append(i_cell[garde])
    print(f"  polygones {start:,}–{min(start + PAQUET, len(geoms)):,} : {garde.sum():,} morceaux")

morceaux = pd.DataFrame({"cell": np.concatenate(morceaux_cell) if morceaux_cell else np.array([], dtype=int),
                         "geom": np.concatenate(morceaux_geom) if morceaux_geom else np.array([], dtype=object)})

aire_canopee = np.zeros(len(carreaux))
n_morceaux = morceaux["cell"].value_counts()
seuls = morceaux[morceaux["cell"].isin(n_morceaux.index[n_morceaux == 1])]
aire_canopee[seuls["cell"].to_numpy()] = shapely.area(seuls["geom"].to_numpy())

multiples = morceaux[morceaux["cell"].isin(n_morceaux.index[n_morceaux > 1])]
for cell, grp in multiples.groupby("cell"):
    aire_canopee[cell] = shapely.union_all(grp["geom"].to_numpy()).area

print(f"\nCanopée dans les carreaux retenus : {aire_canopee.sum() / 1e6:,.2f} km² (chevauchements fusionnés)")
print(f"{(aire_canopee > 0).sum():,} carreaux sur {len(carreaux):,} contiennent de la canopée")

  polygones 0–129,164 : 270,212 morceaux

Canopée dans les carreaux retenus : 1,017.73 km² (chevauchements fusionnés)
48,829 carreaux sur 56,460 contiennent de la canopée


## Scores 0–1 et version seuillée

- `canopee_part = aire_canopee / aire_reference`, où `aire_reference` est l'aire du
  carreau (ou, avec `EMPRISE_CANOPEE`, l'aire du carreau couverte par les données).
  Bornée à [0, 1] pour absorber les écarts d'arrondi géométrique.
- `canopee_seuil = 1` si `canopee_part ≥ SEUIL`, sinon `0`.
- Les carreaux dont la couverture par les données est inférieure à `COUVERTURE_MIN`
  reçoivent des scores vides.

In [38]:
if EMPRISE_CANOPEE is not None:
    emprise = read_any(EMPRISE_CANOPEE).union_all()
    aire_ref = shapely.area(shapely.intersection(cell_geoms, emprise))
else:
    aire_ref = aire_carreau.copy()
couverture = aire_ref / aire_carreau

with np.errstate(invalid="ignore", divide="ignore"):
    part = np.clip(aire_canopee / aire_ref, 0, 1)
valide = couverture >= COUVERTURE_MIN
part[~valide] = np.nan

stats = pd.DataFrame({
    "aire_canopee_m2": aire_canopee.round(1),
    "couverture_donnees": couverture.round(4),
    "canopee_part": part.round(4),
})

base = carreaux.drop(columns="geometry")
geom = carreaux.geometry
centres = geom.centroid
coords = pd.DataFrame({"x_lv95": centres.x.round(1), "y_lv95": centres.y.round(1)})

couche_part = gpd.GeoDataFrame(pd.concat([base, stats, coords], axis=1), geometry=geom, crs=TARGET_CRS)

couche_seuil = couche_part.copy()
couche_seuil["seuil"] = SEUIL
couche_seuil["canopee_seuil"] = pd.array(np.where(valide, (part >= SEUIL).astype(int), pd.NA), dtype="Int8")
couche_seuil = couche_seuil.drop(columns="canopee_part")

print(couche_part["canopee_part"].describe().round(3).to_string())
print(f"\nCarreaux à 1 avec un seuil de {SEUIL:.0%} : {int(couche_seuil['canopee_seuil'].sum()):,} "
      f"sur {int(couche_seuil['canopee_seuil'].notna().sum()):,} ({couche_seuil['canopee_seuil'].mean():.1%})")
if (~valide).any():
    print(f"{(~valide).sum():,} carreaux sans score (couverture des données < {COUVERTURE_MIN:.0%})")

count    56460.000
mean         0.451
std          0.376
min          0.000
25%          0.078
50%          0.376
75%          0.867
max          1.000

Carreaux à 1 avec un seuil de 50% : 23,880 sur 56,460 (42.3%)


## Export

- `canopee_carreau200.gpkg`, deux couches :
  - `canopee_part` : score continu 0–1 ;
  - `canopee_seuil` : score binaire 0/1 (le seuil utilisé est dans la colonne `seuil`) ;
- un Parquet (géométrie incluse) et un CSV (sans géométrie, avec le centre LV95) par couche.

Les attributs d'origine du carreau (`GRID_ID`, population, emplois…) sont conservés.

In [39]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
gpkg = OUT_DIR / f"{OUT_STEM}.gpkg"
if gpkg.exists():
    gpkg.unlink()                       # évite d'empiler des couches d'un run précédent

seuil_tag = f"seuil{int(round(SEUIL * 100))}"
for nom, couche, stem in [("canopee_part", couche_part, f"{OUT_STEM}_part"),
                          ("canopee_seuil", couche_seuil, f"{OUT_STEM}_{seuil_tag}")]:
    couche.to_file(gpkg, layer=nom, driver="GPKG", engine="pyogrio")
    couche.to_parquet(OUT_DIR / f"{stem}.parquet")
    couche.drop(columns="geometry").to_csv(OUT_DIR / f"{stem}.csv", index=False)

for f in sorted(OUT_DIR.glob(f"{OUT_STEM}*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
print("Couches du GPKG :", [n for n, _ in pyogrio.list_layers(gpkg)])

sorties_canopee/canopee_carreau200.gpkg  (31.0 Mo)
sorties_canopee/canopee_carreau200_part.csv  (4.8 Mo)
sorties_canopee/canopee_carreau200_part.parquet  (2.8 Mo)
sorties_canopee/canopee_carreau200_seuil50.csv  (4.8 Mo)
sorties_canopee/canopee_carreau200_seuil50.parquet  (2.7 Mo)
Couches du GPKG : ['canopee_part', 'canopee_seuil']


## Contrôle rapide

In [40]:
print(pd.cut(couche_part["canopee_part"], [-0.001, 0, 0.1, 0.25, 0.5, 0.75, 1],
             labels=["0", "0–10 %", "10–25 %", "25–50 %", "50–75 %", "75–100 %"]).value_counts().sort_index().to_string())
print()
print(couche_seuil["canopee_seuil"].value_counts(dropna=False).sort_index().to_string())

canopee_part
0            7697
0–10 %       7658
10–25 %      7491
25–50 %      9734
50–75 %      6842
75–100 %    17038

canopee_seuil
0    32580
1    23880


In [54]:
# === Charge automobile : raster 10 m -> carroyage 200 m (moyenne) ===
# Chaque pixel est rattaché au carreau de 200 m qui contient son CENTRE (les pixels à
# cheval sur une limite vont donc entièrement d'un côté ou de l'autre). Les deux grilles
# ne sont pas alignées : le rattachement se fait par calcul d'indice sur la grille
# régulière de 200 m, un centre pile sur une limite n'est jamais compté deux fois.
# Le raster est lu par bandes de lignes, limité à l'emprise des carreaux retenus.
import rasterio
from rasterio.windows import Window, from_bounds
from pyproj import Transformer

CHARGE_TIF = "/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/output/PL_output/density_all.tif"   # à adapter
CHARGE_EXTRA_NODATA = []       # valeurs à traiter comme « pas de donnée » en plus du nodata déclaré
CHARGE_STEM = "charge_automobile"
BANDE_LIGNES = 512
CHARGE_ECHELLE = 1000  

# --- géométrie de la grille 200 m, déduite des carreaux eux-mêmes ---
b200 = carreaux.geometry.bounds
pas200 = float(np.median(b200.maxx - b200.minx))
gx0, gy0 = float(b200.minx.min()), float(b200.miny.min())
gi = np.round((b200.minx - gx0) / pas200).astype(int).to_numpy()
gj = np.round((b200.miny - gy0) / pas200).astype(int).to_numpy()
assert np.allclose(b200.minx - gx0, gi * pas200, atol=1) and np.allclose(b200.miny - gy0, gj * pas200, atol=1), \
    "Carroyage 200 m non régulier."
gncol, gnrow = gi.max() + 1, gj.max() + 1
lut200 = np.full(gnrow * gncol, -1, dtype=np.int64)
lut200[gj * gncol + gi] = np.arange(len(carreaux))

CHARGE_Q_CLIP = 0.99           # quantile d'écrêtage des pixels 10 m (None pour désactiver)

n200 = len(carreaux)
idx_l, val_l = [], []          # indice de carreau et valeur de chaque pixel valide retenu

with rasterio.open(CHARGE_TIF) as src:
    if src.crs is None:
        raise ValueError("Le raster n'a pas de SCR déclaré.")
    print(f"{CHARGE_TIF}\n  SCR : {src.crs.to_string()} | pixel : {src.res[0]:g} x {src.res[1]:g} | "
          f"nodata : {src.nodata} | type : {src.dtypes[0]}")
    to_target = None if src.crs == rasterio.crs.CRS.from_user_input(TARGET_CRS) else \
        Transformer.from_crs(src.crs, TARGET_CRS, always_xy=True)
    marge = 2 * max(src.res)
    bb = gpd.GeoSeries([shapely.box(*carreaux.total_bounds)], crs=TARGET_CRS).to_crs(src.crs).total_bounds
    win = from_bounds(bb[0] - marge, bb[1] - marge, bb[2] + marge, bb[3] + marge, transform=src.transform)
    win = win.round_offsets().round_lengths().intersection(Window(0, 0, src.width, src.height))
    if abs(pas200 / src.res[0] - round(pas200 / src.res[0])) > 1e-6:
        print(f"  ⚠️ {pas200:g} m n'est pas un multiple de la taille de pixel : le nombre de pixels par carreau varie.")

    cols = np.arange(win.col_off, win.col_off + win.width) + 0.5
    for r0 in range(win.row_off, win.row_off + win.height, BANDE_LIGNES):
        h = min(BANDE_LIGNES, win.row_off + win.height - r0)
        a = src.read(1, window=Window(win.col_off, r0, win.width, h)).astype("float64")
        bad = ~np.isfinite(a)
        if src.nodata is not None and not np.isnan(src.nodata):
            bad |= a == src.nodata
        for v in CHARGE_EXTRA_NODATA:
            bad |= a == v
        rr, cc = np.nonzero(~bad)
        if rr.size == 0:
            continue
        x, y = src.transform * (cols[cc], rr + r0 + 0.5)          # centres des pixels
        if to_target is not None:
            x, y = to_target.transform(x, y)
        pi = np.floor((np.asarray(x) - gx0) / pas200).astype(np.int64)
        pj = np.floor((np.asarray(y) - gy0) / pas200).astype(np.int64)
        dans = (pi >= 0) & (pi < gncol) & (pj >= 0) & (pj < gnrow)
        idx = np.full(rr.size, -1, dtype=np.int64)
        idx[dans] = lut200[pj[dans] * gncol + pi[dans]]
        k = idx >= 0
        idx_l.append(idx[k])
        val_l.append(a[rr[k], cc[k]])
    n_theo = (pas200 / src.res[0]) ** 2

idx_all = np.concatenate(idx_l) if idx_l else np.empty(0, dtype=np.int64)
val_all = np.concatenate(val_l) if val_l else np.empty(0)
del idx_l, val_l

# --- écrêtage des pixels 10 m au quantile haut (calculé sur les pixels rattachés à un carreau) ---
if CHARGE_Q_CLIP is not None and val_all.size:
    seuil_clip = float(np.quantile(val_all, CHARGE_Q_CLIP))
    ecrete = val_all > seuil_clip
    val_clip = np.minimum(val_all, seuil_clip)
    print(f"  Écrêtage pixels au q{CHARGE_Q_CLIP*100:g} : seuil = {seuil_clip:.6g} "
          f"({ecrete.sum():,} pixels sur {val_all.size:,} ramenés au seuil)")
else:
    seuil_clip = np.nan
    ecrete = np.zeros(val_all.size, dtype=bool)
    val_clip = val_all

# --- agrégation par carreau de 200 m ---
n_pix = np.bincount(idx_all, minlength=n200)
n_ecretes = np.bincount(idx_all, weights=ecrete, minlength=n200).astype(np.int64)
somme_brute = np.bincount(idx_all, weights=val_all, minlength=n200)
somme = np.bincount(idx_all, weights=val_clip, minlength=n200)

with np.errstate(invalid="ignore", divide="ignore"):
    moyenne_brute = somme_brute / n_pix         # NaN si aucun pixel valide
    moyenne = somme / n_pix                     # moyenne des pixels écrêtés

moyenne_echelle = moyenne * CHARGE_ECHELLE
couche_charge = gpd.GeoDataFrame(
    pd.concat([carreaux.drop(columns="geometry"),
               pd.DataFrame({"n_pix_10m": n_pix,
                             "n_pix_ecretes": n_ecretes,
                             "couverture_charge": (n_pix / n_theo).round(4),
                             "charge_automobile": moyenne_echelle,
                             "charge_automobile_brute": moyenne_brute,
                             "charge_echelle": CHARGE_ECHELLE,
                             "charge_seuil_clip": seuil_clip}),
               coords], axis=1),
    geometry=carreaux.geometry, crs=TARGET_CRS)
OUT_DIR.mkdir(parents=True, exist_ok=True)
gpkg_ch = OUT_DIR / f"{CHARGE_STEM}.gpkg"
if gpkg_ch.exists():
    gpkg_ch.unlink()
couche_charge.to_file(gpkg_ch, layer=CHARGE_STEM, driver="GPKG", engine="pyogrio")
couche_charge.to_parquet(OUT_DIR / f"{CHARGE_STEM}.parquet")
couche_charge.drop(columns="geometry").to_csv(OUT_DIR / f"{CHARGE_STEM}.csv", index=False)

print(f"\n{(n_pix > 0).sum():,} carreaux de 200 m avec une valeur sur {n200:,} "
      f"({(n_pix == n_theo).sum():,} entièrement couverts, {n_theo:g} pixels)")
for f in sorted(OUT_DIR.glob(f"{CHARGE_STEM}.*")):
    print(f"{f}  ({f.stat().st_size / 1e6:,.1f} Mo)")
couche_charge[["n_pix_10m", "couverture_charge", "charge_automobile"]].describe()


/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/Data/output/PL_output/density_all.tif
  SCR : EPSG:2056 | pixel : 10 x 10 | nodata : nan | type : float32
  Écrêtage pixels au q99 : seuil = 0.000139847 (198,853 pixels sur 19,885,278 ramenés au seuil)

51,213 carreaux de 200 m avec une valeur sur 56,460 (47,944 entièrement couverts, 400 pixels)
sorties_canopee/charge_automobile.csv  (7.1 Mo)
sorties_canopee/charge_automobile.gpkg  (16.0 Mo)
sorties_canopee/charge_automobile.parquet  (2.8 Mo)


,n_pix_10m,couverture_charge,charge_automobile
count,56460.000000,56460.000000,51213.000000
mean,352.201169,0.880503,0.003620
std,124.728435,0.311821,0.013796
min,0.000000,0.000000,0.000000
25%,400.000000,1.000000,0.000000
50%,400.000000,1.000000,0.000000
75%,400.000000,1.000000,0.000582
max,400.000000,1.000000,0.139847


## Sources à citer

- Carroyage : « Source : Système d'information du territoire à Genève (SITG),
  extrait en date du 31.03.2025 », avec mention du traitement : « Agrégation de la
  canopée réalisée sur la base de données issues du SITG » (CGU SITG, art. 9).
- Périmètre : SITG, idem.
- Canopée : à compléter selon la source et la licence de la couche utilisée.